# E1 remaining baselines: fusion and ExtraTrees

Run the fusion and deterministic ExtraTrees baselines against the exact completed
E1-H0 split. Attach the E0 high-level-cache Kaggle dataset. Attaching downloaded
H0 results is optional and adds paired reporting. Kaggle exposes uploaded datasets
as already-unpacked folders; this notebook deliberately does not search for or
unpack ZIP files.

Large Hugging Face tensors and the derived ExtraTrees feature cache stay under
`/tmp`, outside `/kaggle/working`. Only compact configs, logs, predictions,
checkpoints, and packaged results are written under `/kaggle/working`.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import torch

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO_DIR = WORK / "hls-surrogate-lab"
RESULTS_DIR = WORK / "results"
CONFIG_DIR = WORK / "configs"

REPO_URL = "https://github.com/brios-polimi/hls-surrogate-lab.git"
# This is the code commit recorded by the completed E1-H0 run and E0 release.
REPO_REF = "68829040d8c32f3218409fbdf78d302dcf141d30"
TENSOR_REPO_ID = "BrendanRios/wa-hls4ml-tensors-hierarchical"
TENSOR_REVISION = "1999fbea0187d4cac859f37dd20488fd63eb8860"
E0_RELEASE_ID = "vitis-a31-coarsearch1-hierarchy2-vocab-2026-09-04"
EXPECTED_SPLIT_SHA256 = "bef130022e6b8c08c1b9fdaf6a59535c7a0197d11b9556745866678ac8e9637f"
EXPECTED_SPLIT_SIZES = {"train": 10685, "validation": 2251, "test": 2357, "exemplar": 886}

# Optional H0 results and the required high-level-cache dataset are already
# unpacked by Kaggle under /kaggle/input. Set overrides only if discovery is ambiguous.
H0_RUN_DIR_OVERRIDE = None
HIGH_LEVEL_CACHE_OVERRIDE = None
PREVIOUS_RESULTS_ROOT = None

# Use separate sessions so the 11-hour fusion budget cannot crowd out ExtraTrees.
# After fusion completes, change this to ["extra_trees"] for the CPU baseline.
ACTIVE_RUNS = ["fusion"]
TRAIN_BUDGETS = {"fusion": "11h"}
USE_DDP = True
EFFECTIVE_GLOBAL_BATCH = 16
SEED = 42
ARCHIVE_COUNT = 31
STUDY_ID = f"e1_remaining_baselines_a{ARCHIVE_COUNT}"
ARCHIVE_NAME = f"hls_surrogate_lab_{STUDY_ID}_results"
HF_CACHE_DIR = Path("/tmp/wa_hls4ml_hierarchy_hf_cache")
ET_CACHE_DIR = Path("/tmp/wa_hls4ml_extra_trees_cache")
KERNEL_TYPES = [
    "2layer", "3layer", "conv1d", "conv2d",
    "dense_latency", "dense_resource", "rule4ml",
]

def assert_commit(value, name):
    assert re.fullmatch(r"[0-9a-fA-F]{40,64}", value), (name, value)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert_commit(REPO_REF, "REPO_REF")
assert_commit(TENSOR_REVISION, "TENSOR_REVISION")
assert set(ACTIVE_RUNS) <= {"fusion", "extra_trees"}, ACTIVE_RUNS
assert INPUT.is_dir(), "Kaggle input mount is unavailable"

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CACHE"] = str(HF_CACHE_DIR / "xet")


## Install dependencies and checkout the immutable E1 code

Kaggle already provides CUDA-enabled PyTorch. The repository checkout is pinned
to the exact commit recorded by H0.


In [ ]:
%pip install -q torch-geometric "huggingface_hub[hf_xet]>=0.32" pyyaml pandas scikit-learn


In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert commit == REPO_REF, (commit, REPO_REF)

sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
from ll_hls4ml.data.high_level import PROCESSED_FEATURE_DIM
from ll_hls4ml.data.vocab import load_vocab
from ll_hls4ml.io.schema import LABEL_KEYS
from ll_hls4ml.models.registry import list_models
from ll_hls4ml.reporting.accounting import split_sha256

assert "hierarchical_high_level_fusion" in set(list_models())
print("hls-surrogate-lab commit:", commit)


## Reconstruct the frozen split and optionally find H0 results

The split is reconstructed from the immutable tensor index and accepted only if
it matches E0's recorded sizes and split hash. An unpacked H0 result is optional:
when found it supplies paired predictions and an additional manifest check.


In [ ]:
from huggingface_hub import hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

index_path = Path(hf_hub_download(
    repo_id=TENSOR_REPO_ID, filename="labels.json", repo_type="dataset",
    revision=TENSOR_REVISION, token=hf_token, cache_dir=str(HF_CACHE_DIR),
))
tensor_index = json.loads(index_path.read_text())
metadata = tensor_index.get("metadata", {})

def archive_number(path):
    return int(Path(path).parts[1].removeprefix("archive_"))

def unique_by_graph_id(paths):
    kept = {}
    for path in sorted(paths):
        kept.setdefault(Path(path).stem, path)
    return sorted(kept.values())

included_archives = set(range(1, ARCHIVE_COUNT + 1))
main_paths = unique_by_graph_id([
    path for path in tensor_index["labels"]
    if Path(path).parts[0] in KERNEL_TYPES
    and archive_number(path) in included_archives
])
exemplar_paths = unique_by_graph_id([
    path for path in tensor_index["labels"]
    if Path(path).parts[0] == "exemplar"
])
split_manifest = {name: [] for name in ("train", "validation", "test")}
for path in main_paths:
    split = str(metadata[path].get("dataset_split", "")).lower()
    split = "validation" if split in {"val", "validation"} else split
    assert split in split_manifest, f"Missing official split for {path}"
    split_manifest[split].append({
        "kernel_family": Path(path).parts[0], "tensor_path": path,
    })
split_manifest["exemplar"] = [
    {"kernel_family": "exemplar", "tensor_path": path}
    for path in exemplar_paths
]
assert {name: len(rows) for name, rows in split_manifest.items()} == EXPECTED_SPLIT_SIZES
assert split_sha256(split_manifest) == EXPECTED_SPLIT_SHA256

def valid_h0_run(path):
    required = [
        path / "resolved_config.json", path / "split_manifest.json",
        path / "predictions.csv", path / "summary.json",
    ]
    if not all(item.is_file() for item in required):
        return False
    config = json.loads((path / "resolved_config.json").read_text())
    return (
        config.get("model") == "hierarchical"
        and config.get("seed") == SEED
        and config.get("archive_count") == ARCHIVE_COUNT
        and config.get("tensor_source_revision") == TENSOR_REVISION
        and config.get("ll_hls4ml_git", {}).get("commit") == REPO_REF
    )

if H0_RUN_DIR_OVERRIDE is not None:
    H0_RUN_DIR = Path(H0_RUN_DIR_OVERRIDE)
    assert valid_h0_run(H0_RUN_DIR), H0_RUN_DIR
else:
    candidates = sorted({
        item.parent
        for root in (INPUT, RESULTS_DIR)
        if root.exists()
        for item in root.rglob("resolved_config.json")
        if item.parent.name == f"h0_a{ARCHIVE_COUNT}_seed{SEED}"
        and valid_h0_run(item.parent)
    })
    if candidates:
        fingerprints = {
            (file_sha256(path / "split_manifest.json"), file_sha256(path / "predictions.csv"))
            for path in candidates
        }
        assert len(fingerprints) == 1, f"Conflicting H0 result copies: {candidates}"
        H0_RUN_DIR = candidates[0]
    else:
        H0_RUN_DIR = None

if H0_RUN_DIR is not None:
    h0_manifest = json.loads((H0_RUN_DIR / "split_manifest.json").read_text())
    assert split_sha256(h0_manifest) == EXPECTED_SPLIT_SHA256
    assert h0_manifest == split_manifest, "Reconstructed manifest differs from H0"
all_paths = [row["tensor_path"] for rows in split_manifest.values() for row in rows]
assert len(all_paths) == len(set(all_paths)) == sum(EXPECTED_SPLIT_SIZES.values())
SPLIT_MANIFEST_PATH = CONFIG_DIR / f"official_a{ARCHIVE_COUNT}_manifest.json"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_MANIFEST_PATH.write_text(json.dumps(split_manifest, indent=2))
if H0_RUN_DIR is not None:
    assert file_sha256(SPLIT_MANIFEST_PATH) == file_sha256(H0_RUN_DIR / "split_manifest.json")
    print("Optional H0 source found:", H0_RUN_DIR)
else:
    print("No H0 results attached; paired comparisons will be skipped.")
print("Frozen split:", EXPECTED_SPLIT_SIZES, EXPECTED_SPLIT_SHA256)


## Locate and validate the uploaded E0 high-level cache

Discovery searches nested, already-unpacked Kaggle input folders. Candidate
files are accepted only when their schema contains the complete frozen H0
membership. A cache may additionally contain archive-32 main-family entries;
the manifest prevents those entries from entering training or evaluation.
Unrelated `.pt` tensors and checkpoints are never loaded.


In [ ]:
HIGH_LEVEL_CACHE_PATH = None
high_level_cache = None
if "fusion" in ACTIVE_RUNS:
    if HIGH_LEVEL_CACHE_OVERRIDE is not None:
        candidates = [Path(HIGH_LEVEL_CACHE_OVERRIDE)]
    else:
        patterns = ("high_level_cache.pt", "*high*level*.pt", "*high_level*cache*.pth")
        candidates = sorted({
            path
            for pattern in patterns
            for path in INPUT.rglob(pattern)
            if path.is_file() and "archive_" not in path.as_posix()
        })
    assert candidates, (
        "No uploaded high-level cache found. Attach the E0 release/cache Kaggle "
        "dataset or set HIGH_LEVEL_CACHE_OVERRIDE."
    )
    valid = []
    wanted = set(all_paths)
    rejected = []
    for candidate in candidates:
        try:
            payload = torch.load(candidate, map_location="cpu", weights_only=False)
            samples = payload.get("samples") if isinstance(payload, dict) else None
            reasons = []
            if not isinstance(samples, dict):
                reasons.append("missing samples dictionary")
            else:
                missing = wanted - set(samples)
                extras = set(samples) - wanted
                unexpected_extras = {
                    path for path in extras
                    if len(Path(path).parts) != 3
                    or Path(path).parts[0] not in KERNEL_TYPES
                    or Path(path).parts[1] != "archive_32"
                }
                if missing:
                    reasons.append(f"missing {len(missing)} frozen samples")
                if unexpected_extras:
                    reasons.append(
                        f"contains {len(unexpected_extras)} non-archive-32 extras"
                    )
            if payload.get("processed_feature_dim") != PROCESSED_FEATURE_DIM:
                reasons.append("processed feature dimension differs")
            if payload.get("release_id") not in {None, E0_RELEASE_ID}:
                reasons.append(f"wrong release_id={payload.get('release_id')}")
            if reasons:
                rejected.append((str(candidate), reasons))
            else:
                valid.append((candidate, payload, file_sha256(candidate), len(extras)))
        except Exception as error:
            rejected.append((str(candidate), [f"load failed: {type(error).__name__}"]))
    assert valid, f"No cache exactly matches E0/H0 membership. Rejected: {rejected}"
    fewest_extras = min(item[3] for item in valid)
    valid = [item for item in valid if item[3] == fewest_extras]
    assert len({item[2] for item in valid}) == 1, (
        f"Multiple non-identical equally specific caches found: {[str(x[0]) for x in valid]}"
    )
    (
        HIGH_LEVEL_CACHE_PATH, high_level_cache,
        HIGH_LEVEL_CACHE_SHA256, HIGH_LEVEL_CACHE_EXTRA_COUNT,
    ) = valid[0]
    print("High-level cache:", HIGH_LEVEL_CACHE_PATH)
    print("High-level cache SHA-256:", HIGH_LEVEL_CACHE_SHA256)
    print("Ignored cache entries outside E0:", HIGH_LEVEL_CACHE_EXTRA_COUNT)
else:
    print("Fusion inactive: high-level cache discovery skipped.")


## Download the exact CDFG tensor cohort outside `/kaggle/working`

Only manifest members are requested from the immutable Hugging Face revision.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download, login, snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
remote_files = set(HfApi().list_repo_files(
    TENSOR_REPO_ID, repo_type="dataset", revision=TENSOR_REVISION, token=hf_token,
))
missing_remote = sorted(set(all_paths) - remote_files)
assert not missing_remote, f"Manifest tensors missing remotely; first: {missing_remote[:5]}"

patterns = sorted({
    f"{Path(path).parts[0]}/{Path(path).parts[1]}/*.pt" for path in all_paths
})
TENSOR_DIR = Path(snapshot_download(
    repo_id=TENSOR_REPO_ID, repo_type="dataset", revision=TENSOR_REVISION,
    token=hf_token, cache_dir=str(HF_CACHE_DIR),
    allow_patterns=[*patterns, "labels.json", "vocab.json"],
))
VOCAB_PATH = TENSOR_DIR / "vocab.json"
assert VOCAB_PATH.is_file()
missing_local = [path for path in all_paths if not (TENSOR_DIR / path).is_file()]
assert not missing_local, f"Incomplete local snapshot; first: {missing_local[:5]}"
print("Tensor snapshot:", TENSOR_DIR)
print("Validated local tensors:", len(all_paths))


## Configure resumable fusion training

The fusion model uses the exact E0/H0 manifest. H0 predictions are added only
when available, for paired reporting. Resume discovery searches unpacked Kaggle
inputs for the run directory and never unpacks ZIPs.


In [ ]:
GPU_COUNT = torch.cuda.device_count()
if "fusion" in ACTIVE_RUNS:
    assert GPU_COUNT >= 1, "Enable a Kaggle GPU accelerator for fusion"
    if USE_DDP:
        assert EFFECTIVE_GLOBAL_BATCH % GPU_COUNT == 0
PRECISION = "bf16" if GPU_COUNT and torch.cuda.is_bf16_supported() else "float32"

FUSION_EXPERIMENT = f"fusion_a{ARCHIVE_COUNT}_seed{SEED}"
fusion_config = {
    "experiment_name": FUSION_EXPERIMENT,
    "model": "hierarchical_high_level_fusion",
    "tensor_dir": str(TENSOR_DIR),
    "tensor_source_revision": TENSOR_REVISION,
    "vocab_path": str(VOCAB_PATH),
    "high_level_cache": str(HIGH_LEVEL_CACHE_PATH) if HIGH_LEVEL_CACHE_PATH else None,
    "split_manifest_path": str(SPLIT_MANIFEST_PATH),
    "require_complete_split_manifest": True,
    "results_dir": str(RESULTS_DIR),
    "checkpoint_dir": str(RESULTS_DIR / FUSION_EXPERIMENT / "checkpoints"),
    "kernel_types": KERNEL_TYPES,
    "study_id": STUDY_ID,
    "protocol_id": "official_in_distribution",
    "archive_count": ARCHIVE_COUNT,
    "distributed_world_size": GPU_COUNT if USE_DDP else 1,
    "seed": SEED,
    "family_balanced_sampling": False,
    "effective_global_batch": EFFECTIVE_GLOBAL_BATCH,
    "batch_size": EFFECTIVE_GLOBAL_BATCH // (GPU_COUNT if USE_DDP else 1),
    "num_workers": 2, "worker_tmpdir": "/tmp", "pin_memory": True,
    "prefetch_factor": 2, "thread_prefetch": False,
    "precision": PRECISION, "epochs": 400, "patience": 20,
    "learning_rate": 1e-3, "weight_decay": 1e-4,
    "hidden_dim": 64, "num_layers": 3, "heads": 1, "dropout": 0.15,
    "high_level_encoder": "gatv2", "use_global_features": True,
    "use_context": True, "context_mode": "core", "split_heads": True,
    "hurdle_heads": True, "hurdle_prediction_mode": "threshold",
    "loss": "log_huber_hurdle", "log_huber_delta": 0.35,
    "hurdle_classification_weight": 0.25,
    "early_stopping_metric": "smape", "gradient_clip_norm": 1.0,
    "lr_scheduler_patience": 8, "lr_scheduler_factor": 0.5,
    "min_learning_rate": 1e-6, "checkpoint_interval": 5, "verbose": 2,
}
if H0_RUN_DIR is not None:
    fusion_config["baseline_predictions_path"] = str(H0_RUN_DIR / "predictions.csv")

def fusion_signature():
    return {
        "repo_ref": REPO_REF, "tensor_revision": TENSOR_REVISION,
        "split_sha256": EXPECTED_SPLIT_SHA256,
        "high_level_cache_sha256": HIGH_LEVEL_CACHE_SHA256,
        "high_level_cache_extra_ignored": HIGH_LEVEL_CACHE_EXTRA_COUNT,
        **{key: fusion_config[key] for key in (
            "model", "seed", "archive_count", "effective_global_batch",
            "batch_size", "precision", "distributed_world_size", "epochs",
            "patience", "learning_rate", "weight_decay", "hidden_dim",
            "num_layers", "heads", "dropout", "high_level_encoder",
            "use_global_features", "use_context", "context_mode", "split_heads",
            "hurdle_heads", "loss", "checkpoint_interval",
        )},
    }

def previous_root():
    root = Path(PREVIOUS_RESULTS_ROOT) if PREVIOUS_RESULTS_ROOT else INPUT
    assert root.is_dir(), root
    return root

def prepare_fusion():
    run_dir = RESULTS_DIR / FUSION_EXPERIMENT
    expected = fusion_signature()
    matches = sorted({
        item.parent for item in previous_root().rglob("notebook_resume_signature.json")
        if item.parent.name == FUSION_EXPERIMENT
    })
    if not run_dir.exists() and matches:
        signatures = [json.loads((path / "notebook_resume_signature.json").read_text()) for path in matches]
        assert all(value == expected for value in signatures), "Incompatible prior fusion run"
        fingerprints = {file_sha256(path / "notebook_resume_signature.json") for path in matches}
        assert len(fingerprints) == 1, f"Conflicting prior fusion runs: {matches}"
        shutil.copytree(matches[0], run_dir)
        print("Imported previous fusion run:", matches[0])
    run_dir.mkdir(parents=True, exist_ok=True)
    signature_path = run_dir / "notebook_resume_signature.json"
    if signature_path.is_file():
        assert json.loads(signature_path.read_text()) == expected
    signature_path.write_text(json.dumps(expected, indent=2))
    payload = dict(fusion_config)
    backup = Path(fusion_config["checkpoint_dir"]) / f"{FUSION_EXPERIMENT}_backup.pt"
    if backup.is_file():
        payload["resume_checkpoint_path"] = str(backup)
    path = CONFIG_DIR / f"{FUSION_EXPERIMENT}.json"
    path.write_text(json.dumps(payload, indent=2))
    return path

RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import shlex

TRAIN_SCRIPT = REPO_DIR / "scripts/train.py"
run_environment = os.environ.copy()
run_environment["PYTHONPATH"] = str(REPO_DIR / "src")
run_environment["LL_HLS4ML_TQDM"] = "0"
run_environment["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"

def package_results():
    archive = Path(shutil.make_archive(str(WORK / ARCHIVE_NAME), "zip", root_dir=RESULTS_DIR))
    print("Updated result archive:", archive)
    return archive

def run_and_stream(command, log_path):
    print("$", shlex.join(command), flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", buffering=1) as log:
        process = subprocess.Popen(
            command, cwd=REPO_DIR, env=run_environment,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            log.write(line)
        return process.wait()

def run_fusion():
    if "fusion" not in ACTIVE_RUNS:
        print("Fusion inactive")
        return
    config_path = prepare_fusion()
    run_dir = RESULTS_DIR / FUSION_EXPERIMENT
    summary_path = run_dir / "summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        if summary.get("resolved_config", {}).get("evaluation_checkpoint_path") is None:
            print("Fusion already completed normally")
            package_results()
            return
    if USE_DDP and GPU_COUNT > 1:
        train_command = [
            sys.executable, "-m", "torch.distributed.run", "--standalone",
            f"--nproc_per_node={GPU_COUNT}", str(TRAIN_SCRIPT),
            "--config", str(config_path),
        ]
    else:
        train_command = [sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path)]
    command = ["timeout", "--signal=INT", "--kill-after=5m", TRAIN_BUDGETS["fusion"], *train_command]
    try:
        return_code = run_and_stream(command, run_dir / "training.log")
        if return_code == 0:
            assert summary_path.is_file()
            return
        assert return_code in {124, 130, 137}, f"Unexpected fusion exit: {return_code}"
        checkpoint_dir = Path(fusion_config["checkpoint_dir"])
        checkpoint = checkpoint_dir / f"{FUSION_EXPERIMENT}_checkpoint.pt"
        if not checkpoint.is_file():
            checkpoint = checkpoint_dir / f"{FUSION_EXPERIMENT}_backup.pt"
        assert checkpoint.is_file(), "Time limit expired before a checkpoint existed"
        evaluation = [
            sys.executable, str(TRAIN_SCRIPT), "--config", str(config_path),
            "--evaluate-checkpoint", str(checkpoint),
        ]
        assert run_and_stream(evaluation, run_dir / "evaluation.log") == 0
    finally:
        package_results()

run_fusion()


## Run the exact-membership ExtraTrees baseline

This uses deterministic graph-summary and synthesis-context features, not
adjacency or message passing. Its hurdle mode is selected on validation only.
The large derived feature table is cached under `/tmp`.


In [ ]:
def run_extra_trees():
    if "extra_trees" not in ACTIVE_RUNS:
        print("ExtraTrees inactive")
        return
    import numpy as np
    import pandas as pd
    from ll_hls4ml.data.dataset import HeteroGraphDataset
    from hierarchical_cpu_baseline import (
        HurdleRegressor, arrays, bootstrap_delta, choose_mode,
        feature_frame, feature_sets, metric_rows,
    )

    experiment = f"extra_trees_a{ARCHIVE_COUNT}_seed{SEED}"
    output = RESULTS_DIR / experiment
    summary_path = output / "summary.csv"
    if summary_path.is_file():
        print("ExtraTrees already completed:", output)
        return
    output.mkdir(parents=True, exist_ok=True)
    ET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    dataset = HeteroGraphDataset(
        TENSOR_DIR, types=[*KERNEL_TYPES, "exemplar"], silent=False,
    )
    vocabulary, _, _ = load_vocab(VOCAB_PATH)
    needed = set(all_paths)
    frame = feature_frame(
        dataset, needed, len(vocabulary), ET_CACHE_DIR / "tabular_features.pkl",
    )
    indexed = frame.set_index("tensor_path")
    columns = feature_sets(frame)["core_context"]
    paths = {
        split: [row["tensor_path"] for row in split_manifest[split]]
        for split in split_manifest
    }
    train_x, train_y = arrays(indexed, paths["train"], columns)
    val_x, val_y = arrays(indexed, paths["validation"], columns)
    started = time.perf_counter()
    model = HurdleRegressor("extra_trees", SEED).fit(train_x, train_y)
    fit_seconds = time.perf_counter() - started
    mode = choose_mode(model, val_x, val_y)

    metrics = []
    predictions = []
    comparisons = []
    h0 = (
        __import__("pandas").read_csv(H0_RUN_DIR / "predictions.csv").set_index(
            ["split", "tensor_path"]
        )
        if H0_RUN_DIR is not None else None
    )
    for split in ("validation", "test", "exemplar"):
        split_x, target = arrays(indexed, paths[split], columns)
        prediction = model.predict_modes(split_x)[mode]
        families = indexed.loc[paths[split], "kernel_family"].to_numpy(str)
        metrics.extend(metric_rows(
            experiment, "extra_trees", "core_context", len(paths["train"]),
            split, families, prediction, target,
        ))
        for row_index, tensor_path in enumerate(paths[split]):
            row = {
                "experiment": experiment, "model": "extra_trees",
                "feature_set": "core_context", "split": split,
                "tensor_path": tensor_path, "kernel_family": families[row_index],
                "selected_hurdle_mode": mode,
            }
            for target_index, label in enumerate(LABEL_KEYS):
                row[f"target_{label}"] = float(target[row_index, target_index])
                row[f"prediction_{label}"] = float(prediction[row_index, target_index])
            predictions.append(row)
        if h0 is not None and split in {"test", "exemplar"}:
            neural_rows = h0.loc[[(split, path) for path in paths[split]]]
            neural_prediction = neural_rows[
                [f"prediction_{label}" for label in LABEL_KEYS]
            ].to_numpy(np.float32)
            delta, low, high, win_fraction = bootstrap_delta(
                prediction, neural_prediction, target, SEED,
            )
            comparisons.append({
                "split": split, "extra_trees_minus_h0_smape": delta,
                "ci95_low": low, "ci95_high": high,
                "extra_trees_sample_win_fraction": win_fraction,
                "n_samples": len(paths[split]),
            })

    metric_frame = pd.DataFrame(metrics)
    metric_frame.to_csv(output / "metrics.csv", index=False)
    pd.DataFrame(predictions).to_csv(output / "predictions.csv", index=False)
    pd.DataFrame(comparisons).to_csv(output / "paired_h0_comparisons.csv", index=False)
    summary = (
        metric_frame[metric_frame.kernel_family == "all"]
        .groupby(["experiment", "model", "feature_set", "train_size", "split"], as_index=False)
        .agg(macro_smape=("smape", "mean"), macro_r2=("r2", "mean"))
    )
    summary.to_csv(summary_path, index=False)
    resolved = {
        "experiment": experiment, "model": "extra_trees", "seed": SEED,
        "source_h0_run": str(H0_RUN_DIR) if H0_RUN_DIR else None,
        "source_split_sha256": EXPECTED_SPLIT_SHA256,
        "tensor_revision": TENSOR_REVISION, "repo_ref": REPO_REF,
        "train_size": len(paths["train"]), "validation_size": len(paths["validation"]),
        "test_size": len(paths["test"]), "exemplar_size": len(paths["exemplar"]),
        "feature_set": "core_context", "feature_count": len(columns),
        "feature_interpretation": "deterministic graph summaries and synthesis context; no adjacency",
        "fit_seconds": fit_seconds, "selected_hurdle_mode": mode,
        "extra_trees": {"n_estimators": 300, "min_samples_leaf": 2, "max_features": 0.8},
    }
    (output / "resolved_config.json").write_text(json.dumps(resolved, indent=2) + "\n")
    print(summary.to_string(index=False))
    print("ExtraTrees output:", output)

run_extra_trees()
package_results()


## Review and download

Fusion writes standard neural result files, including exact paired deltas versus
H0. ExtraTrees writes its own metrics, predictions, paired bootstrap comparison,
timing, and resolved configuration.


In [ ]:
import pandas as pd
for candidate in (
    RESULTS_DIR / FUSION_EXPERIMENT / "summary.json",
    RESULTS_DIR / f"extra_trees_a{ARCHIVE_COUNT}_seed{SEED}" / "summary.csv",
):
    if candidate.is_file():
        print("\n", candidate)
        if candidate.suffix == ".json":
            print(json.dumps(json.loads(candidate.read_text()), indent=2)[:4000])
        else:
            display(pd.read_csv(candidate))

archive = package_results()
from IPython.display import FileLink
display(FileLink(str(archive)))


## Interpretation guardrails

- Fusion and ExtraTrees must retain H0's split hash and exact membership.
- Fusion is time-capped until it reaches normal early stopping; resume from the unpacked prior run directory in a later session.
- ExtraTrees sees pooled graph statistics and synthesis context, but no adjacency. It is a non-neural strength check, not a topology model.
- Do not add archive 32. It is outside the E0 release boundary because dense_resource is incomplete upstream.
- Preserve the H0 tensor revision and the validated E0 high-level-cache hash with all reported results.
